# T040 — Knowledge-tracing benchmark: results

**Ticket:** T040 (KK-60). **Decision record:** [ADR 0007](../../docs/architecture/decisions/0007-knowledge-tracing-model.md) (Proposed — the product owner decides).

This notebook reads the results saved by `kt_benchmark.py --mode full` and shows them as tables and charts. It trains nothing and runs in a few seconds. To reproduce the numbers, run the script (see [README](README.md)); the run logs every training run to MLflow with its seed, data source and code version.

**What is compared.** DKT (baseline), SAKT, SAINT and AKT from pyKT, and SAINT+ from `kuizi_brain` (pyKT has no SAINT+). All models have the same width (64), the same optimiser, the same windows and the same early stopping; no model was tuned individually.

**How.** Students are split into 5 folds with a fixed seed; three folds are run, each trained on the other students (10 % of them held out for early stopping) and tested on students the model has never seen. Every answer of a test student except the first is predicted exactly once, so all models are scored on the same predictions. Numbers are mean ± standard deviation over the three folds.

**Charts.** Each model has its own marker shape and line style, and values are written on the charts, so no chart relies on colour alone.

In [ ]:
# Load the saved results (concept: reproducible benchmark, T040)
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODE = "full"
candidates = [Path.cwd() / "results", Path.cwd() / "benchmarks" / "results", Path.cwd() / "ml" / "benchmarks" / "results"]
RESULTS = next(p for p in candidates if (p / f"t040_{MODE}_meta.json").exists())
read = lambda name: pd.read_csv(RESULTS / f"t040_{MODE}_{name}.csv")
meta = json.loads((RESULTS / f"t040_{MODE}_meta.json").read_text())
runs, slices, calib, latency, gate, history = (read(n) for n in ("runs", "slices", "calibration", "latency", "gate", "history"))
MAIN = meta["config"]["assist2009"]["main_length"]

LABELS = {"dkt": "DKT", "sakt": "SAKT", "saint": "SAINT", "akt": "AKT", "saint_plus": "SAINT+",
          "saint_plus_notime": "SAINT+ without time", "saint_plus_fairtime": "SAINT+ time / multiplier"}
MARKERS = {"dkt": "o", "sakt": "s", "saint": "^", "akt": "D", "saint_plus": "v", "saint_plus_notime": "P",
           "saint_plus_fairtime": "X"}
STYLES = {"dkt": "-", "sakt": "--", "saint": "-.", "akt": ":", "saint_plus": (0, (5, 1, 1, 1)),
          "saint_plus_notime": (0, (1, 1)), "saint_plus_fairtime": (0, (3, 1, 1, 1, 1, 1))}
ORDER = ["dkt", "sakt", "saint", "akt", "saint_plus", "saint_plus_notime", "saint_plus_fairtime"]
pm = lambda m, s: f"{m:.4f} ± {s:.4f}" if pd.notna(s) else f"{m:.4f}"
pd.set_option("display.width", 200)

env = meta["environment"]
print(f"Mode {meta['mode']}: {meta['training_runs']} training runs, training {meta['training_wall_seconds'] / 60:.1f} min, "
      f"total {meta['total_wall_seconds'] / 60:.1f} min on {env['cpu_count']} CPU cores "
      f"({meta['workers']} workers x {meta['threads_per_worker']} thread)")
print(f"Python {env['python']}, PyTorch {env['torch']}, pykt-toolkit {env['pykt_toolkit']}, CUDA available: {env['cuda_available']}")
print(f"Code: commit {meta['code']['git_commit'][:10]} on {meta['code']['git_branch']} (uncommitted changes: {meta['code']['git_dirty']}), "
      f"code sha256 {meta['code']['code_sha256'][:12]}")
print(f"Data: {meta['data_sources']['assist2009']['name']}, sha256 {meta['data_sha256']['assist2009'][:12]}")
print(f"Folds: {meta['n_folds']} by student (seed {meta['fold_seed']}), folds run: {meta['config']['folds']}; "
      f"up to {meta['config']['max_epochs']} epochs, early stopping patience {meta['config']['patience']}")
print("ASSISTments 2017:", meta["assist2017"])

## 1. Main result — ASSISTments 2009, windows of 100 answers

AUC: how often the model ranks a right answer above a wrong one (0.5 = coin toss). ECE: average gap between the predicted chance and the observed share of right answers (0 = perfectly calibrated). Accuracy uses a 0.5 threshold.

In [ ]:
# Main table (concepts: AUC, accuracy, calibration; production ticket T040, gate rule T047)
a = runs[(runs.dataset == "assist2009") & (runs.max_len == MAIN)]
g = a.groupby("model")
table = pd.DataFrame({
    "AUC": g.auc.mean(), "AUC sd": g.auc.std(), "accuracy": g.accuracy.mean(), "accuracy sd": g.accuracy.std(),
    "ECE": g.ece.mean(), "ECE sd": g.ece.std(), "folds": g.fold.nunique(), "parameters": g.parameters.first(),
    "best epoch (mean)": g.best_epoch.mean(), "training s (mean)": g.train_seconds.mean(),
}).sort_values("AUC", ascending=False)
shown = pd.DataFrame({
    "model": [LABELS[m] for m in table.index],
    "AUC": [pm(r.AUC, r["AUC sd"]) for _, r in table.iterrows()],
    "accuracy": [pm(r.accuracy, r["accuracy sd"]) for _, r in table.iterrows()],
    "ECE": [pm(r.ECE, r["ECE sd"]) for _, r in table.iterrows()],
    "parameters": table.parameters.astype(int).map("{:,}".format).values,
    "best epoch": table["best epoch (mean)"].round(1).values,
    "training s": table["training s (mean)"].round(0).astype(int).values,
})
print(f"Test predictions per fold: {int(a.n_predictions.mean()):,} answers from {int(a.test_students.mean())} students")
shown

In [ ]:
# Chart: AUC and ECE per model with the spread over folds (markers + values, not colour)
order = list(table.index[::-1])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, metric, label in [(axes[0], "AUC", "AUC (higher is better)"), (axes[1], "ECE", "ECE (lower is better)")]:
    for i, m in enumerate(order):
        mean, sd = table.loc[m, metric], table.loc[m, f"{metric} sd"]
        ax.errorbar(mean, i, xerr=sd, fmt=MARKERS[m], color="black", ms=8, capsize=4)
        ax.annotate(f"{mean:.3f}", (mean, i), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9)
    ax.set_xlabel(label)
    ax.grid(axis="x", ls=":", lw=0.6)
axes[0].set_yticks(range(len(order)), [LABELS[m] for m in order])
axes[0].set_ylim(-0.6, len(order) - 0.4)
fig.suptitle(f"ASSISTments 2009, windows of {MAIN} answers: mean over folds, bars = ± 1 standard deviation")
fig.tight_layout()
plt.show()

## 2. Sequence-length sensitivity

The same models trained and tested with windows of 50, 100 and 200 answers. Each test answer is predicted from at least half a window of earlier answers when the student has them, so a longer window means more history per prediction.

In [ ]:
# Sequence length (concept: how much history the model reads; T042 plans 512 with a sliding window, T048 builds the window)
a = runs[runs.dataset == "assist2009"]
lengths = sorted(a.max_len.unique())
models = [m for m in ORDER if m in set(a.model) and a[a.model == m].max_len.nunique() > 1]
s = a.groupby(["model", "max_len"]).auc.agg(["mean", "std"])
display_tab = pd.DataFrame({f"L = {L}": [pm(*s.loc[(m, L)]) for m in models] for L in lengths},
                           index=[LABELS[m] for m in models])
fig, ax = plt.subplots(figsize=(8, 4.5))
for m in models:
    y = [s.loc[(m, L), "mean"] for L in lengths]
    e = [s.loc[(m, L), "std"] for L in lengths]
    ax.errorbar(lengths, y, yerr=e, marker=MARKERS[m], ls=STYLES[m], color="black", ms=7, capsize=3, label=LABELS[m])
    ax.annotate(LABELS[m], (lengths[-1], y[-1]), textcoords="offset points", xytext=(8, 0), va="center", fontsize=9)
ax.set_xticks(lengths)
ax.set_xlim(lengths[0] - 15, lengths[-1] + 60)
ax.set_xlabel("window length (answers)")
ax.set_ylabel("AUC on unseen students")
ax.set_title("AUC against window length (mean ± 1 sd over folds)")
ax.legend(loc="lower left", fontsize=8, ncol=2)
ax.grid(ls=":", lw=0.6)
fig.tight_layout()
plt.show()
display_tab

## 3. Calibration

One panel per model. Points on the dotted diagonal mean that when the model says 70 %, students are right about 70 % of the time. The size of each bin is written next to its point (thousands of answers), because sparse bins move a lot.

In [ ]:
# Reliability diagrams pooled over folds (concept: calibration, T040 / T047 / T050)
c = calib[(calib.dataset == "assist2009") & (calib.max_len == MAIN)].copy()
c["pw"], c["ow"] = c.predicted * c.answers, c.observed * c.answers
pooled = c.groupby(["model", "bin"]).agg(pw=("pw", "sum"), ow=("ow", "sum"), answers=("answers", "sum")).reset_index()
pooled["predicted"], pooled["observed"] = pooled.pw / pooled.answers, pooled.ow / pooled.answers
models = [m for m in ORDER if m in set(pooled.model)]
fig, axes = plt.subplots(1, len(models), figsize=(3.1 * len(models), 3.4), sharey=True)
for ax, m in zip(axes, models):
    p = pooled[pooled.model == m].sort_values("predicted")
    ax.plot([0, 1], [0, 1], ls=":", color="grey", lw=1)
    ax.plot(p.predicted, p.observed, marker=MARKERS[m], ls=STYLES[m], color="black", ms=6)
    for _, r in p.iterrows():
        ax.annotate(f"{r.answers / 1000:.0f}k", (r.predicted, r.observed), textcoords="offset points", xytext=(4, -10), fontsize=7)
    ece = table.loc[m, "ECE"] if m in table.index else np.nan
    ax.set_title(f"{LABELS[m]}\nECE {ece:.3f}", fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("predicted chance")
axes[0].set_ylabel("observed share right")
fig.tight_layout()
plt.show()

## 4. Slices and the T047 gate rule against DKT

Slices by position in the student's history: answers 2–10 are the cold start (where the IRT estimator of T043 will help), answers 51 and later are students with a long history. ASSISTments has no accommodation data, so the accommodation slices are measured on the simulator in section 5.

The T047 gate rule says a challenger is better than the champion only if its AUC is at least 0.005 higher, no slice loses more than 0.01 AUC, and ECE is not worse by more than 0.01. Differences are paired by fold (same test students).

In [ ]:
# AUC per slice (concept: fairness and cold-start slices, T047)
sl = slices[(slices.dataset == "assist2009") & (slices.max_len == MAIN)]
slice_order = ["answers 2-10", "answers 11-50", "answers 51+"]
t = sl.groupby(["model", "slice"]).agg(auc=("auc", "mean"), sd=("auc", "std"), ece=("ece", "mean"),
                                       n=("n_predictions", "mean")).reset_index()
models = [m for m in ORDER if m in set(t.model)]
auc_tab = pd.DataFrame({s_: [pm(*t[(t.model == m) & (t.slice == s_)][["auc", "sd"]].iloc[0]) for m in models]
                        for s_ in slice_order}, index=[LABELS[m] for m in models])
n_tab = t[t.model == "dkt"].set_index("slice").n.reindex(slice_order).round(0).astype(int)
print("Test answers per fold in each slice:", n_tab.to_dict())
auc_tab

In [ ]:
# The T047 gate, fold-paired, against the DKT baseline
gt = gate[(gate.dataset == "assist2009")].copy()
gt["challenger"] = gt.challenger.map(LABELS)
gt[["max_len", "challenger", "folds", "delta_auc_mean", "delta_auc_std", "delta_ece_mean", "worst_slice",
    "worst_slice_delta_auc", "passes_t047_gate"]].round(4).sort_values(["max_len", "delta_auc_mean"], ascending=[True, False])

## 5. Time features and accommodations

SAINT+ adds two time signals to SAINT: how long the previous answer took (elapsed time) and the gap since the previous answer (lag time). ASSISTments 2009 has elapsed time but no timestamps, so on it SAINT+ uses elapsed time only. The simulator has both, and it has students with an official time multiplier (x1.33 or x1.5) and an assistive mode, so it is where the accommodation slices are checked. The simulator is synthetic: it tells us whether a model can be unfair to students with extra time, not how large the effect is for real students.

In [ ]:
# Does elapsed time help on real data? Paired by fold (concept: SAINT+ time embeddings, T042)
a = runs[(runs.dataset == "assist2009") & (runs.max_len == MAIN)].pivot_table(index="fold", columns="model", values=["auc", "ece"])
if "saint_plus_notime" in a["auc"]:
    d = a["auc"]["saint_plus"] - a["auc"]["saint_plus_notime"]
    print(f"ASSISTments 2009: SAINT+ {pm(a['auc']['saint_plus'].mean(), a['auc']['saint_plus'].std())}, "
          f"without time {pm(a['auc']['saint_plus_notime'].mean(), a['auc']['saint_plus_notime'].std())}; "
          f"paired gain from elapsed time {d.mean():+.4f} ± {d.std():.4f} AUC")

In [ ]:
# Simulator: overall and accommodation slices (concept: fairness for accommodations, T047, T148)
sim = runs[runs.dataset == "simulated"]
sim_tab = None
if len(sim):
    ss = slices[slices.dataset == "simulated"]
    cols = ["no accommodation", "extra time (x1.33 or x1.5)", "assistive mode"]
    models = [m for m in ORDER if m in set(sim.model)]
    rows = []
    for m in models:
        r = sim[sim.model == m]
        row = {"model": LABELS[m], "AUC": pm(r.auc.mean(), r.auc.std()), "ECE": pm(r.ece.mean(), r.ece.std())}
        for c_ in cols:
            x = ss[(ss.model == m) & (ss.slice == c_)]
            row[f"AUC {c_}"] = f"{x.auc.mean():.4f}"
            row[f"ECE {c_}"] = f"{x.ece.mean():.4f}"
        x0 = ss[(ss.model == m) & (ss.slice == cols[0])].set_index("fold")
        x1 = ss[(ss.model == m) & (ss.slice == cols[1])].set_index("fold")
        row["AUC gap extra time - none"] = f"{(x1.auc - x0.auc).mean():+.4f}"
        rows.append(row)
    n = ss[(ss.model == "dkt")].groupby("slice").n_predictions.mean().round(0).astype(int)
    print("Simulated test answers per fold in each slice:", {k: int(n[k]) for k in cols if k in n})
    sim_tab = pd.DataFrame(rows).set_index("model")
sim_tab

In [ ]:
# Chart: AUC per accommodation group on the simulator (markers + values)
if len(sim):
    fig, ax = plt.subplots(figsize=(9, 4))
    groups = cols
    for i, m in enumerate(models):
        y = [ss[(ss.model == m) & (ss.slice == g_)].auc.mean() for g_ in groups]
        ax.plot(range(len(groups)), y, marker=MARKERS[m], ls=STYLES[m], color="black", ms=7, label=LABELS[m])
        ax.annotate(LABELS[m], (len(groups) - 1, y[-1]), textcoords="offset points", xytext=(8, 0), va="center", fontsize=8)
    ax.set_xticks(range(len(groups)), groups)
    ax.set_xlim(-0.2, len(groups) + 0.6)
    ax.set_ylabel("AUC (simulated students)")
    ax.set_title("AUC per accommodation group, simulator, mean over folds")
    ax.grid(ls=":", lw=0.6)
    ax.legend(fontsize=8, loc="lower left", ncol=2)
    fig.tight_layout()
    plt.show()

## 6. CPU inference latency

One forward pass over a full window predicts the next answer of every student in the batch, as the online service will (T046, T048). Latency is the median over many passes on this machine's CPU, with PyTorch using one thread or four. GPU latency is pending: there is no GPU before the cloud (S8+), and no number is invented for it.

In [ ]:
# Latency table (concept: inference cost, T046 serving, T048 sequence builder)
lat = latency[latency.device == "cpu"].copy()
lat["model"] = lat.model.map(LABELS)
pivot = lat.pivot_table(index=["model", "max_len"], columns=["batch", "threads"], values="ms_per_prediction")
pivot.columns = [f"batch {int(b)}, {int(t)} thread{'s' if t > 1 else ''}" for b, t in pivot.columns]
gpu = latency[latency.device == "gpu"]
print(f"GPU: {gpu.note.iloc[0] if len(gpu) else 'not measured'}")
print("ms per prediction (median); one prediction = next-answer probability for one student")
pivot.round(3)

In [ ]:
# Chart: ms per prediction against window length, batch 1 (a single student request), one thread
lat1 = latency[(latency.device == "cpu") & (latency.batch == 1) & (latency.threads == 1)]
fig, ax = plt.subplots(figsize=(8, 4.2))
for m in [m for m in ORDER if m in set(lat1.model)]:
    r = lat1[lat1.model == m].sort_values("max_len")
    ax.plot(r.max_len, r.ms_per_prediction, marker=MARKERS[m], ls=STYLES[m], color="black", ms=7, label=LABELS[m])
    ax.annotate(f"{LABELS[m]} {r.ms_per_prediction.iloc[-1]:.1f} ms", (r.max_len.iloc[-1], r.ms_per_prediction.iloc[-1]),
                textcoords="offset points", xytext=(8, 0), va="center", fontsize=8)
ax.set_xticks(sorted(lat1.max_len.unique()))
ax.set_xlim(lat1.max_len.min() - 15, lat1.max_len.max() + 110)
ax.set_xlabel("window length (answers)")
ax.set_ylabel("ms per prediction, batch 1, 1 thread")
ax.set_title("CPU latency of one student request (median)")
ax.grid(ls=":", lw=0.6)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

## 7. Training curves

Validation AUC after each epoch (fold 0, windows of 100). Early stopping keeps the best epoch; a curve that is still rising at the last epoch means the model might gain from longer training.

In [ ]:
# Validation AUC per epoch (concept: early stopping on held-out students)
h = history[(history.dataset == "assist2009") & (history.max_len == MAIN) & (history.fold == 0)]
fig, ax = plt.subplots(figsize=(8, 4))
for m in [m for m in ORDER if m in set(h.model)]:
    r = h[h.model == m]
    ax.plot(r.epoch, r.valid_auc, marker=MARKERS[m], ls=STYLES[m], color="black", ms=6, label=LABELS[m])
    ax.annotate(LABELS[m], (r.epoch.iloc[-1], r.valid_auc.iloc[-1]), textcoords="offset points", xytext=(6, 0), va="center", fontsize=8)
ax.set_xlabel("epoch")
ax.set_ylabel("validation AUC")
ax.set_xlim(0.5, h.epoch.max() + 2.5)
ax.grid(ls=":", lw=0.6)
ax.legend(fontsize=8, loc="lower right")
ax.set_title("Validation AUC per epoch, fold 0")
fig.tight_layout()
plt.show()
stopped_at_max = runs[(runs.best_epoch == meta["config"]["max_epochs"])]
print(f"Runs whose best epoch was the last allowed epoch ({meta['config']['max_epochs']}): {len(stopped_at_max)} of {len(runs)}",
      stopped_at_max.groupby(["dataset", "model"]).size().to_dict())

## Findings

(to be written from the results)